
# DLC → trials → linearized TMaze coordinates

This notebook does the following, using **only** DLC / trim / trial files:

1. Load `dlc_trim_info.csv`, the TMaze `*_TMaze_trials.csv`, and the DLC `*_filtered.csv`.
2. Use the `most_robust_bodypart` from `dlc_trim_info.csv` to pick the DLC columns.
3. Clean the 2D coordinates.
4. Adjust trial timing so that frames are expressed relative to the **trimmed video** start frame.
5. Linearize the coordinates along the TMaze centerlines for each trial.
6. Make a few diagnostic plots for each trial type to visually verify the linearization.
7. Optionally save the linearized trial coordinates in the **same directory** as the `dlc_trim_info.csv` / DLC.


In [ ]:

import numpy as np
import pandas as pd
import os
import glob

import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# Plot style (optional)
plt.rcParams['figure.dpi'] = 100

# Global thresholds
likelihood_thresh = 0.9
max_jump_pixels = 5        # for cleaning jumps
smoothing_sigma = 1.0      # for Gaussian smoothing


In [ ]:

def col_of(dlc_df, bodypart, coord):
    """Return MultiIndex column for given bodypart and coord in {'x','y','likelihood'}."""
    cs = [c for c in dlc_df.columns
          if len(c) == 3
          and c[1] == bodypart
          and str(c[2]).strip().lower() == coord]
    if not cs:
        raise ValueError(f"Missing column for bodypart={bodypart}, coord={coord}."
                         f" Available head levels: {dlc_df.columns[:5]}")
    return cs[0]


In [ ]:

def process_location_2D(location,
                        likelihood,
                        likelihood_threshold=0.9,
                        max_jump=5,
                        smoothing_sigma=1.0):
    """
    Clean 2D DLC coordinates.

    Parameters
    ----------
    location:   (N, 2) array of [x, y]
    likelihood: (N,) array of DLC likelihoods for that bodypart
    """

    loc = np.asarray(location, dtype=float)
    lh  = np.asarray(likelihood, dtype=float).squeeze()

    if loc.ndim != 2 or loc.shape[1] != 2:
        raise ValueError("location must be (N, 2) array")
    if loc.shape[0] != lh.shape[0]:
        raise ValueError("location and likelihood must have same length")

    # 1) Mask low-likelihood frames
    bad_lh = lh < likelihood_threshold
    loc[bad_lh, :] = np.nan

    # 2) Interpolate NaNs separately for x and y
    for dim in range(2):
        s = pd.Series(loc[:, dim])
        loc[:, dim] = s.interpolate(limit_direction='both').values

    # 3) Detect large 2D jumps
    dx = np.diff(loc[:, 0])
    dy = np.diff(loc[:, 1])
    dist = np.sqrt(dx**2 + dy**2)

    bad_jump = dist > max_jump

    # mark both sides of bad jumps as bad
    bad_mask = np.zeros(loc.shape[0], dtype=bool)
    bad_mask[1:] |= bad_jump
    bad_mask[:-1] |= bad_jump

    good_idx = np.where(~bad_mask)[0]
    if len(good_idx) >= 2:
        bad_idx = np.flatnonzero(bad_mask)
        # interpolate in x and y over bad indices
        for dim in range(2):
            loc[bad_idx, dim] = np.interp(
                bad_idx,
                good_idx,
                loc[good_idx, dim]
            )

    # 4) Smooth each dimension
    cleaned = np.empty_like(loc)
    for dim in range(2):
        cleaned[:, dim] = gaussian_filter1d(loc[:, dim],
                                            sigma=smoothing_sigma)

    return cleaned


In [ ]:

def _project_onto_segment(p, a, b):
    """Project point p onto segment a-b."""
    ab = b - a
    ab2 = np.dot(ab, ab)
    if ab2 == 0:
        return a, 0.0, np.sum((p - a)**2)
    t = np.dot(p - a, ab) / ab2
    t_clipped = np.clip(t, 0.0, 1.0)
    proj = a + t_clipped * ab
    dist2 = np.sum((p - proj)**2)
    return proj, t_clipped, dist2


def linearize_positions_along_polyline(location, poly_points):
    """
    location    : (N, 2) array of [x, y] positions
    poly_points : (M, 2) array of [x, y] defining the maze centerline

    Returns:
    linear_pos  : (N,) array, 1D position (same units as coords; e.g. pixels or cm)
    """
    loc = np.asarray(location, float)
    poly = np.asarray(poly_points, float)

    # precompute segment lengths and cumulative lengths
    seg_vecs = poly[1:] - poly[:-1]
    seg_len = np.linalg.norm(seg_vecs, axis=1)
    cum_len = np.concatenate([[0.0], np.cumsum(seg_len)])  # length at poly[i]

    N = loc.shape[0]
    linear_pos = np.full(N, np.nan)

    for i in range(N):
        p = loc[i]
        if np.any(np.isnan(p)):
            continue

        best_dist2 = np.inf
        best_s = np.nan  # arc-length along polyline

        for s_idx in range(len(seg_len)):
            a = poly[s_idx]
            b = poly[s_idx + 1]
            _, t, d2 = _project_onto_segment(p, a, b)
            if d2 < best_dist2:
                best_dist2 = d2
                best_s = cum_len[s_idx] + t * seg_len[s_idx]

        linear_pos[i] = best_s

    return linear_pos


# --- TMaze geometry (pixels, adapt to your setup) ---
base      = np.array([103,   0])   # bottom of stem
choice    = np.array([105, 112])   # intersection of the T
left_end  = np.array([  0, 115])   # left arm end
right_end = np.array([210, 110])   # right arm end

# Polylines for each path
poly_BL = np.vstack([base,   choice, left_end])    # base ↔ left
poly_BR = np.vstack([base,   choice, right_end])   # base ↔ right
poly_LR = np.vstack([left_end, choice, right_end]) # left ↔ right

# Trial-type groups (you can adapt names if your CSV uses different labels)
bl_types = {"base_to_left", "left_to_base"}
br_types = {"base_to_right", "right_to_base"}
lr_types = {"left_to_right", "right_to_left"}


def get_linpos_for_trial_type(trial_type, linpos_BL, linpos_BR, linpos_LR):
    """Pick the appropriate linearized trajectory for a given trial_type."""
    if trial_type in bl_types:
        return linpos_BL
    elif trial_type in br_types:
        return linpos_BR
    elif trial_type in lr_types:
        return linpos_LR
    else:
        raise ValueError(f"Unknown trial_type: {trial_type}")


In [ ]:

def load_trim_info(trim_csv_path):
    """Load dlc_trim_info.csv and return a Series with start_frame etc."""
    trim_df = pd.read_csv(trim_csv_path)
    if trim_df.empty:
        raise ValueError("dlc_trim_info.csv is empty.")
    row = trim_df.iloc[0]
    required_cols = {"start_frame", "start_time_seconds", "most_robust_bodypart"}
    missing = required_cols - set(trim_df.columns)
    if missing:
        raise ValueError(f"dlc_trim_info.csv is missing columns: {missing}")
    return row


def load_trials(trials_csv_path):
    """Load TMaze trials CSV; expected to contain start_frame, end_frame, trial_type."""
    trials = pd.read_csv(trials_csv_path)
    required_cols = {"start_frame", "end_frame", "trial_type"}
    missing = required_cols - set(trials.columns)
    if missing:
        raise ValueError(f"Trials CSV is missing columns: {missing}")
    return trials


def load_dlc_for_bodypart(dlc_csv_path, bodypart):
    """
    Load DLC filtered CSV and return raw location and likelihood for the given bodypart.
    """
    dlc_df = pd.read_csv(dlc_csv_path, header=[0, 1, 2])
    x_col = col_of(dlc_df, bodypart, 'x')
    y_col = col_of(dlc_df, bodypart, 'y')
    l_col = col_of(dlc_df, bodypart, 'likelihood')

    x = pd.to_numeric(dlc_df[x_col], errors='coerce').to_numpy()
    y = pd.to_numeric(dlc_df[y_col], errors='coerce').to_numpy()
    lh = pd.to_numeric(dlc_df[l_col], errors='coerce').to_numpy()

    location_raw = np.column_stack([x, y])
    return location_raw, lh


In [ ]:

def adjust_trials_for_trim(trials, trim_start_frame, n_frames_trimmed=None):
    """
    Re-express trial start/end frames relative to the trimmed video.

    New columns:
        - trim_start
        - trim_end
    """
    trials_adj = trials.copy()
    trials_adj["trim_start"] = trials_adj["start_frame"] - int(trim_start_frame)
    trials_adj["trim_end"]   = trials_adj["end_frame"]   - int(trim_start_frame)

    if n_frames_trimmed is not None:
        trials_adj["trim_start"] = trials_adj["trim_start"].clip(0, n_frames_trimmed - 1)
        trials_adj["trim_end"]   = trials_adj["trim_end"].clip(0, n_frames_trimmed - 1)

    return trials_adj


def build_trimmed_location(location_raw, likelihood, trim_start_frame,
                           likelihood_threshold=0.9,
                           max_jump=5,
                           smoothing_sigma=1.0):
    """
    Clean full DLC location and then slice to trimmed coordinates.

    Returns
    -------
    location_clean_full : (N, 2)
    location_clean_trim : (N_trim, 2), where frame 0 == trim_start_frame of original
    """
    location_clean_full = process_location_2D(
        location_raw,
        likelihood,
        likelihood_threshold=likelihood_threshold,
        max_jump=max_jump,
        smoothing_sigma=smoothing_sigma,
    )
    trim_start_frame = int(trim_start_frame)
    if trim_start_frame < 0 or trim_start_frame >= location_clean_full.shape[0]:
        raise ValueError("trim_start_frame outside range of DLC frames.")
    location_clean_trim = location_clean_full[trim_start_frame:]
    return location_clean_full, location_clean_trim


In [ ]:

def linearize_trials(location_clean_trim,
                     trials_adj,
                     save=False,
                     trim_info_path=None,
                     out_basename="TMaze_linearized_trials",
                     max_trials_per_type_for_plots=3):
    """
    Linearize TMaze coordinates for each trial and optionally save results.

    Parameters
    ----------
    location_clean_trim : (N_trim, 2)
        Cleaned location in trimmed coordinates (frame 0 == trimmed video start).
    trials_adj : DataFrame
        Must contain columns: trim_start, trim_end, trial_type.
    save : bool
        If True, save a CSV in the same directory as `trim_info_path`.
    trim_info_path : str or None
        Path to dlc_trim_info.csv (used to infer directory for saving).
    out_basename : str
        Base name of the output CSV.
    max_trials_per_type_for_plots : int
        Number of example trials per type to show in the diagnostic plots.

    Returns
    -------
    lin_df : pd.DataFrame
        Long-format table with columns:
            trial_index, trial_type, frame_in_trial, frame_global_trim, linpos
    linpos_BL, linpos_BR, linpos_LR : np.ndarray
        Full linearized trajectories for BL/BR/LR paths.
    """
    loc = np.asarray(location_clean_trim, float)
    if loc.ndim != 2 or loc.shape[1] != 2:
        raise ValueError("location_clean_trim must be (N_trim, 2)")

    # Full-series linearizations
    linpos_BL = linearize_positions_along_polyline(loc, poly_BL)
    linpos_BR = linearize_positions_along_polyline(loc, poly_BR)
    linpos_LR = linearize_positions_along_polyline(loc, poly_LR)

    rows = []
    for idx, tr in trials_adj.iterrows():
        ttype = tr["trial_type"]
        start = int(tr["trim_start"])
        end   = int(tr["trim_end"])
        if start < 0 or end >= loc.shape[0] or end <= start:
            continue

        lin_full = get_linpos_for_trial_type(ttype, linpos_BL, linpos_BR, linpos_LR)
        seg = lin_full[start:end+1]

        for k, val in enumerate(seg):
            rows.append({
                "trial_index": idx,
                "trial_type": ttype,
                "frame_in_trial": k,
                "frame_global_trim": start + k,
                "linpos": float(val),
            })

    lin_df = pd.DataFrame(rows)

    # --- Diagnostic plots ---
    unique_types = trials_adj["trial_type"].unique()
    for ttype in unique_types:
        tr_idxs = trials_adj.index[trials_adj["trial_type"] == ttype].tolist()
        if not tr_idxs:
            continue

        tr_idxs = tr_idxs[:max_trials_per_type_for_plots]
        for tid in tr_idxs:
            tr = trials_adj.loc[tid]
            start = int(tr["trim_start"])
            end   = int(tr["trim_end"])
            if start < 0 or end >= loc.shape[0] or end <= start:
                continue

            coords = loc[start:end+1]
            seg_lin = lin_df[lin_df["trial_index"] == tid]["linpos"].to_numpy()

            fig, axes = plt.subplots(1, 2, figsize=(10, 4))

            # 2D path
            axes[0].plot(loc[:, 0], loc[:, 1], alpha=0.2, label="all (trim)")
            axes[0].plot(coords[:, 0], coords[:, 1], '-o', label=f"trial {tid}")
            axes[0].scatter([base[0], choice[0], left_end[0], right_end[0]],
                            [base[1], choice[1], left_end[1], right_end[1]],
                            marker='x', s=50, label="landmarks")
            axes[0].set_title(f"{ttype} – trial {tid} path (trim coords)")
            axes[0].set_xlabel("x")
            axes[0].set_ylabel("y")
            axes[0].legend()

            # Linearized trace
            axes[1].plot(seg_lin, '-o')
            axes[1].set_title(f"{ttype} – trial {tid} linearized")
            axes[1].set_xlabel("frame in trial")
            axes[1].set_ylabel("linear position (a.u.)")
            axes[1].grid(True)

            plt.tight_layout()
            plt.show()

    # --- Optional save ---
    if save:
        if trim_info_path is None:
            raise ValueError("trim_info_path must be provided when save=True.")
        out_dir = os.path.dirname(os.path.abspath(trim_info_path))
        out_csv = os.path.join(out_dir, out_basename + ".csv")
        lin_df.to_csv(out_csv, index=False)
        print("Saved linearized trial coordinates to:", out_csv)

    return lin_df, linpos_BL, linpos_BR, linpos_LR


In [ ]:
# === Example usage ===
# Edit these three paths for your session:

trim_info_path  = r"/media/toor/T7Shield/Mouse1637/Round1.5/TMaze/2024-06-10T17_53_09/dlc_trim_info.csv"
trials_csv_path = f"/media/toor/T7Shield/Mouse1637/Round1.5/TMaze/2024-06-10T17_53_09/dlc/*TMaze_trials.csv"
dlc_csv_path    = r"/path/to/your_filtered_dlc.csv"

trim_info = load_trim_info(trim_info_path)
trials    = load_trials(trials_csv_path)
bodypart  = trim_info["most_robust_bodypart"]

# location_raw, likelihood = load_dlc_for_bodypart(dlc_csv_path, bodypart)
# location_clean_full, location_clean_trim = build_trimmed_location(
#     location_raw,
#     likelihood,
#     trim_start_frame=trim_info["start_frame"],
#     likelihood_threshold=likelihood_thresh,
#     max_jump=max_jump_pixels,
#     smoothing_sigma=smoothing_sigma,
# )
#
# trials_adj = adjust_trials_for_trim(
#     trials,
#     trim_start_frame=trim_info["start_frame"],
#     n_frames_trimmed=location_clean_trim.shape[0],
# )
#
# lin_df, linpos_BL, linpos_BR, linpos_LR = linearize_trials(
#     location_clean_trim,
#     trials_adj,
#     save=True,                # set False if you don't want to save yet
#     trim_info_path=trim_info_path,
#     out_basename="TMaze_linearized_trials"
# )
